# Define you own problem

Apart from giving you access to already implemented problem, Quantum Launcher allows you to easily define you own problems which will suit your needs better

To define a problem you need two key components, problem initialization and problem formulations in at least one of the two formats used by quantum algorithms, QUBO and Hamiltonian 

## Problem initialization

To complete a problem initialization you need to create a subclass of class problem, when you initialize it it takes as an argument an object of problem instance in any form, e.g. graph matrix if the problem is graph based

In [1]:
from quantum_launcher.base import Problem

class CustomProblem(Problem):
    pass

graph_matrix = [[0,0,1,0],[1,0,1,1],[1,1,0,0],[1,0,0,0]]
pr = CustomProblem(graph_matrix)

You could leave the class declaration like that but in practice is might be worth to define also a set of helper function, such as one generating a random instance, allowing to read instance from file or visualizing the problem with some proposed solution. Below you can find full initialization of the Exact Cover problem:

In [2]:
"""  This module contains the EC class."""
import ast
from collections import defaultdict
from typing import List, Literal, Optional, Set
import networkx as nx

from quantum_launcher.base import Problem


class EC(Problem):
    """
    Class for exact cover problem.

    The exact cover problem is a combinatorial optimization problem that involves finding a subset of a given set
    of elements such that the subset covers all elements and the number of elements in the subset is minimized.
    The class contains an instance of the problem, so it can be passed into Quantum Launcher.

    Attributes:
        onehot (str): The one-hot encoding used for the problem.
        instance (any): The instance of the problem.
        instance_name (str | None): The name of the instance.
        instance_path (str): The path to the instance file.

    """

    def __init__(self, onehot: Literal['exact', 'quadratic'], instance: List[Set[int]] = None,
                 instance_name: str = 'unnamed') -> None:
        super().__init__(instance=instance, instance_name=instance_name)
        self.onehot = onehot

    @property
    def setup(self) -> dict:
        return {
            'onehot': self.onehot,
            'instance_name': self.instance_name
        }

    def _get_path(self) -> str:
        return f'{self.name}@{self.instance_name}@{self.onehot}'

    @staticmethod
    def from_preset(instance_name: str, **kwargs) -> "EC":
        match instance_name:
            case 'micro':
                instance = [{1, 2},
                            {1}]
            case 'toy':
                instance = [{1, 4, 7},
                            {1, 4},
                            {4, 5, 7},
                            {3, 5, 6},
                            {2, 3, 6, 7},
                            {2, 7}]
        return EC(instance=instance, instance_name=instance_name, **kwargs)

    @classmethod
    def from_file(cls, path: str, **kwargs) -> "EC":
        with open(path, 'r', encoding='utf-8') as file:
            read_file = file.read()
        instance = ast.literal_eval(read_file)
        return EC(instance, **kwargs)

    def read_instance(self, instance_path: str):
        with open(instance_path, 'r', encoding='utf-8') as file:
            read_file = file.read()
        self.instance = ast.literal_eval(read_file)

    def visualize(self, marked: Optional[str] = None):
        import matplotlib.pyplot as plt
        G = nx.Graph()
        size = len(self.instance)
        ec = list(map(lambda x: set(map(str, x)), self.instance))
        names = set.union(*ec)
        for i in range(len(ec)):
            G.add_node(i)
        for i in sorted(names):
            G.add_node(i)
        covered = defaultdict(int)
        for node, edges in enumerate(ec):
            for goal_node in edges:
                G.add_edge(node, goal_node)

        edge_colors = []
        for goal_node in G.edges():
            node, str_node = goal_node
            if marked is None:
                edge_colors.append('black')
                continue
            if marked[node] == '1':
                edge_colors.append('red')
                covered[str_node] += 1
            else:
                edge_colors.append('gray')
        color_map = []
        for node in G:
            if isinstance(node, int):
                color_map.append('lightblue')
            elif covered[node] == 0:
                color_map.append('yellow')
            elif covered[node] == 1:
                color_map.append('lightgreen')
            else:
                color_map.append('red')
        pos = nx.bipartite_layout(G, nodes=range(size))
        nx.draw(G, pos, node_color=color_map, with_labels=True,
                edge_color=edge_colors)
        plt.title('Exact Cover Problem Visualization')
        plt.show()

    @staticmethod
    def generate_ec_instance(n: int, m: int, p: float = 0.5, **kwargs) -> "EC":
        graph = nx.bipartite.random_graph(n, m, p)
        right_nodes = [n for n, d in graph.nodes(data=True) if d["bipartite"] == 0]
        instance = [set() for _ in right_nodes]
        for left, right in graph.edges():
            instance[left].add(right)
        return EC(instance=instance, **kwargs)


## Problem Formulation

To add problem formulation you need to define a callable, a function or callable class, and decorate it with the formatter decorator 

The formatter needs to be provided two two argument, the class of a problem you add a formulation for and format in which problem formulation will be returned e.g. "qubo" or "hamiltonian". Additionally, the decorated function needs to take an instance of the specified problem class

In [3]:
from quantum_launcher.base import formatter
import numpy as np

@formatter(problem=CustomProblem, format="qubo")
def get_qubo(problem:CustomProblem):
    q_matrix = [[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]]
    q_matrix = np.array(q_matrix)
    offset = 0
    return q_matrix, offset

The decorated function needs to return a problem formulation in the selected format, for "qubo" the expected format is the qubo matrix followed by offset, and for the hamiltonian formulation it needs to be a SparsePauliOp from the qiskit library, you can check both formulation of Exact Cover below:

In [4]:
from quantum_launcher.problems import problem_initialization as problem
import quantum_launcher.hampy as hampy

@formatter(problem.MaxCut, 'qubo')
def get_maxcut_qubo(problem: problem.MaxCut):
    """ Returns Qubo function """
    n = len(problem.instance)
    Q = np.zeros((n, n))
    for (i, j) in problem.instance.edges:
        Q[i, i] += -1
        Q[j, j] += -1
        Q[i, j] += 1
        Q[j, i] += 1

    return Q, 0

@formatter(problem.MaxCut, 'hamiltonian')
def get_qiskit_hamiltonian(problem: problem.MaxCut):
    ham = None
    n = problem.instance.number_of_nodes()
    for edge in problem.instance.edges():
        if ham is None:
            ham = ~hampy.one_in_n(edge, n)
        else:
            ham += ~hampy.one_in_n(edge, n)
    return ham.hamiltonian.simplify()


A nice convenience the quantum launcher introduces is the fact that you don't need to define formulations in both formats, quantum launcher uses adapters which allow it to transform between problem formulations

Now that you have create a problem initialization and formulation with decorated with @formatter, you can pass an instance of the problem to Quantum Launcher which will automatically find the formulation and convert it to format required by the algorithm

In [5]:
from quantum_launcher.launcher import QuantumLauncher
from quantum_launcher.routines.dwave_routines import DwaveSolver, SimulatedAnnealingBackend

graph_matrix = [[0,0,1,0],[1,0,1,1],[1,1,0,0],[1,0,0,0]]
pr = CustomProblem(graph_matrix)

ql = QuantumLauncher(pr,DwaveSolver(),SimulatedAnnealingBackend())
ql.run()

Result(bitstring=0101, energy=-3.0)